In [29]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [30]:
MY_DIR  = '/data/user/tvaneede/muon_tagging/reproduce_hese7/output'
REF_DIR = '/data/ana/Diffuse/HESE/Pass2/IC86_2015/data_muontag_reprocessed'
RUN     = 'Run00127009'

# Event that passes all cuts in Austin's reference (CausalQTot = 7059 PE)
PASSING_EVENT = 567172

CUT_NAMES = [
    '0_MuonTagExists',
    '1_MuonTagPasses',
    '2_VHEInnerSelfVetoExists',
    '3_VHEInnerSelfVetoPasses',
    '4_CausalQTotPasses',
]

CUT_DESC = {
    '0_MuonTagExists':          'VHESelfVeto (MuonTag) key exists',
    '1_MuonTagPasses':          'VHESelfVeto found a vertex (MuonTag = True)',
    '2_VHEInnerSelfVetoExists': 'Inner VHESelfVeto key exists',
    '3_VHEInnerSelfVetoPasses': 'Inner VHESelfVeto does NOT fire (event is contained)',
    '4_CausalQTotPasses':       'CausalQTot > 6000 PE',
}


def load_cut(base_dir, run, cut_name):
    path = f'{base_dir}/{run}_{cut_name}.h5'
    try:
        with h5py.File(path, 'r') as f:
            ds = f[cut_name][:]
        df = pd.DataFrame(ds)
        df = df.rename(columns={'value': cut_name})
        return df[['Run', 'Event', 'SubEvent', cut_name]]
    except (OSError, KeyError):
        return None

## Cut progression: event counts at each stage

In [31]:
rows = []
for name in CUT_NAMES:
    my  = load_cut(MY_DIR,  RUN, name)
    ref = load_cut(REF_DIR, RUN, name)

    n_my  = len(my)  if my  is not None else 0
    n_ref = len(ref) if ref is not None else 0
    p_my  = int(my[name].sum())  if my  is not None else 0
    p_ref = int(ref[name].sum()) if ref is not None else 0

    rows.append(dict(
        cut=name,
        description=CUT_DESC[name],
        mine_seen=n_my,   mine_pass=p_my,
        ref_seen=n_ref,   ref_pass=p_ref,
    ))

prog = pd.DataFrame(rows).set_index('cut')
prog

,description,mine_seen,mine_pass,ref_seen,ref_pass
cut,,,,,
0_MuonTagExists,VHESelfVeto (MuonTag) key exists,1084,1084,5748,5748
1_MuonTagPasses,VHESelfVeto found a vertex (MuonTag = True),1084,1084,5748,5739
2_VHEInnerSelfVetoExists,Inner VHESelfVeto key exists,1084,917,5739,4893
3_VHEInnerSelfVetoPasses,Inner VHESelfVeto does NOT fire (event is cont...,917,1,4893,2
4_CausalQTotPasses,CausalQTot > 6000 PE,1,1,2,1


## Passing event: cut-by-cut comparison

Event 567172 in Run 127009 passes all 5 cuts in Austin's reference (CausalQTot = 7059 PE).

In [32]:
rows = []
for name in CUT_NAMES:
    my  = load_cut(MY_DIR,  RUN, name)
    ref = load_cut(REF_DIR, RUN, name)

    def get_val(df):
        if df is None:
            return 'not reached'
        row = df[df['Event'] == PASSING_EVENT]
        if len(row) == 0:
            return 'not in file'
        return bool(row[name].values[0])

    rows.append({
        'cut': name,
        'description': CUT_DESC[name],
        'mine': get_val(my),
        'austin': get_val(ref),
        'agree': get_val(my) == get_val(ref),
    })

ev_df = pd.DataFrame(rows).set_index('cut')
print(f'Passing event: Run {RUN}, Event {PASSING_EVENT}\n')
ev_df

Passing event: Run Run00127009, Event 567172



,description,mine,austin,agree
cut,,,,
0_MuonTagExists,VHESelfVeto (MuonTag) key exists,True,True,True
1_MuonTagPasses,VHESelfVeto found a vertex (MuonTag = True),True,True,True
2_VHEInnerSelfVetoExists,Inner VHESelfVeto key exists,True,True,True
3_VHEInnerSelfVetoPasses,Inner VHESelfVeto does NOT fire (event is cont...,True,True,True
4_CausalQTotPasses,CausalQTot > 6000 PE,True,True,True


## Passing event: actual VHESelfVeto and CausalQTot values

Both runs write numeric values to the main `Run00127009.h5` for every event that passes all 5 cuts.

In [33]:
SCALAR_KEYS = ['MuonTag', 'MuonTagTime', 'QTot', 'CausalQTot', 'InteriorCausalQTot',
               'VHEFullSelfVeto', 'VHESelfVetoVertexTime']
VECTOR_KEYS = ['VHESelfVetoVertexPos']  # stored as x, y, z columns


def read_event_values(h5_path, event_id):
    result = {}
    try:
        with h5py.File(h5_path, 'r') as f:
            for key in SCALAR_KEYS:
                if key not in f:
                    result[key] = None
                    continue
                ds = f[key][:]
                row = ds[ds['Event'] == event_id]
                result[key] = float(row['value'][0]) if len(row) else None
            for key in VECTOR_KEYS:
                if key not in f:
                    result[key] = None
                    continue
                ds = f[key][:]
                row = ds[ds['Event'] == event_id]
                result[key] = (float(row['x'][0]), float(row['y'][0]), float(row['z'][0])) if len(row) else None
    except OSError as e:
        print(f'Could not open {h5_path}: {e}')
    return result


ref_vals = read_event_values(f'{REF_DIR}/{RUN}.h5', PASSING_EVENT)
my_vals  = read_event_values(f'{MY_DIR}/{RUN}.h5',  PASSING_EVENT)


def fmt(v):
    if v is None:
        return 'missing'
    if isinstance(v, tuple):
        return f'({v[0]:.2f}, {v[1]:.2f}, {v[2]:.2f})'
    return f'{v:.4f}'


rows = []
for key in SCALAR_KEYS + VECTOR_KEYS:
    a = ref_vals.get(key)
    m = my_vals.get(key)
    rows.append({'variable': key, 'austin': fmt(a), 'mine': fmt(m), 'match': fmt(a) == fmt(m)})

print(f'Passing event: Run {RUN}, Event {PASSING_EVENT}\n')
pd.DataFrame(rows).set_index('variable')

Passing event: Run Run00127009, Event 567172



,austin,mine,match
variable,,,
MuonTag,1.0000,1.0000,True
MuonTagTime,11074.1664,11074.1664,True
QTot,7323.7759,7323.7759,True
CausalQTot,7059.1366,7059.1366,True
InteriorCausalQTot,6788.5602,6608.8486,False
VHEFullSelfVeto,1.0000,1.0000,True
VHESelfVetoVertexTime,11074.1664,11074.1664,True
VHESelfVetoVertexPos,"(248.42, -111.83, -472.10)","(248.42, -111.83, -472.10)",True


In [ ]:
ATSOUFLI_H5 = '/data/user/atsoufli/muon_tagging/output/5th_try/h5/IC86_2015.h5'

with h5py.File(ATSOUFLI_H5, 'r') as f:
    ds = f['CausalQTot'][:]
    row = ds[(ds['Run'] == 127009) & (ds['Event'] == PASSING_EVENT)]
    qtot_atsoufli = float(row['value'][0]) if len(row) else None

print(f'Event Run={RUN}, Event={PASSING_EVENT}')
print()
print(f'  Austin (shrinkydink, icerec.V05-02-00): {ref_vals["CausalQTot"]:.4f} PE')
print(f'  Mine   (shrinkydink, icetray v1.14.0) : {my_vals["CausalQTot"]:.4f} PE')
print(f'  Atsoufli (5th_try pipeline)           : {qtot_atsoufli:.4f} PE')

## Cross-check: atsoufli's VHESelfVeto output\n\nAtsoufli ran a separate VHESelfVeto processing chain (`/data/user/atsoufli/muon_tagging/output/5th_try/h5/`). Comparing CausalQTot for the same event shows whether our processing pipelines agree.

## All shared events: compare VHESelfVeto and VHEInnerSelfVeto outcomes

My test run covers a subset of subruns; Austin processed all subruns.  
We join on `(Event, SubEvent)` to find the subset of events present in both.

In [34]:
cuts_to_compare = [
    '1_MuonTagPasses',
    '2_VHEInnerSelfVetoExists',
    '3_VHEInnerSelfVetoPasses',
]

def build_event_table(base_dir):
    base = load_cut(base_dir, RUN, '0_MuonTagExists')[['Run', 'Event', 'SubEvent', '0_MuonTagExists']]
    for name in cuts_to_compare:
        df = load_cut(base_dir, RUN, name)
        if df is not None:
            base = base.merge(df, on=['Run', 'Event', 'SubEvent'], how='left')
    return base

my_tbl  = build_event_table(MY_DIR).set_index(['Run', 'Event', 'SubEvent'])
ref_tbl = build_event_table(REF_DIR).set_index(['Run', 'Event', 'SubEvent'])

shared = my_tbl.join(ref_tbl, lsuffix='_mine', rsuffix='_ref', how='inner')
print(f'Events in my test run : {len(my_tbl)}')
print(f'Events in Austin ref  : {len(ref_tbl)}')
print(f'Shared events         : {len(shared)}')

for col in ['1_MuonTagPasses', '2_VHEInnerSelfVetoExists', '3_VHEInnerSelfVetoPasses']:
    cm, cr = col + '_mine', col + '_ref'
    if cm in shared.columns and cr in shared.columns:
        agree = (shared[cm].fillna(-1) == shared[cr].fillna(-1)).sum()
        print(f'  {col}: {agree}/{len(shared)} agree')

Events in my test run : 1084
Events in Austin ref  : 5748
Shared events         : 1084
  1_MuonTagPasses: 1084/1084 agree
  2_VHEInnerSelfVetoExists: 1084/1084 agree
  3_VHEInnerSelfVetoPasses: 1084/1084 agree


In [35]:
cols_mine = [c for c in shared.columns if c.endswith('_mine')]
cols_ref  = [c for c in shared.columns if c.endswith('_ref')]

disagree_mask = pd.Series(False, index=shared.index)
for cm, cr in zip(cols_mine, cols_ref):
    disagree_mask = disagree_mask | (shared[cm].fillna(-1) != shared[cr].fillna(-1))

n_disagree = disagree_mask.sum()
print(f'Events with any disagreement: {n_disagree}')
if n_disagree > 0:
    display(shared[disagree_mask])
else:
    print('All shared events agree on every cut — results reproduce Austin correctly.')

Events with any disagreement: 0
All shared events agree on every cut — results reproduce Austin correctly.
